## Feasibility of Enacted Maps to Inexact Contiguity Constraints (Euclidean-M)

Checks whether each district in an enacted plan is feasible under tree-based, distance-based, and DAG-based contiguity constraints using **Euclidean-M** edge weights:

- `eucl(i,j)` if endpoints share the same tract (GEOID20[:11] match)
- `eucl(i,j) + M` if same county, different tract (GEOID20[:5] match)
- `eucl(i,j) + M²` if different county

where M = two-sweep approximate diameter of the point set.

### Data Requirements

1. **Block-level graphs** — stored in `../../data/{state}_block.json`
2. **Block Assignment Files (BAFs)** — run `bash scripts/download_baf.sh` to download and extract, or manually download from:
   - Congressional: https://www2.census.gov/programs-surveys/decennial/rdo/mapping-files/2023/118-congressional-district-bef/cd118.zip
   - State Senate: https://www2.census.gov/programs-surveys/decennial/rdo/mapping-files/2023/2022-state-legislative-bef/sldu_2022.zip
   - State House: https://www2.census.gov/programs-surveys/decennial/rdo/mapping-files/2023/2022-state-legislative-bef/sldl_2022.zip

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT_DIR))

from src.read import read_graph_from_json
from src.params import BaseParams
from src.utils import nearest_node, set_euclidean_M_weights
import networkx as nx
import pandas as pd

In [3]:
# Paths to data
filepath_graphs = str(ROOT_DIR / "data") + "/"
filepath_cd = str(ROOT_DIR / "data" / "baf" / "cd118") + "/"
filepath_ss = str(ROOT_DIR / "data" / "baf" / "sldu_2022") + "/"
filepath_sh = str(ROOT_DIR / "data" / "baf" / "sldl_2022") + "/"

In [ ]:
def get_fips(state):
    return BaseParams.state_fips_mapping[state]


# Number of districts per (state, district_type)
# CD = Congressional, SS = State Senate, SH = State House
number_of_districts = {
    ("IA", "CD"): 4,
    ("IA", "SS"): 50,
    ("IA", "SH"): 100,
}

In [5]:
def read_enacted_districts(G, state, district_type):
    """Read enacted district assignments from a Census BAF file."""
    geoid_to_node = {G.nodes[i]['GEOID20']: i for i in G.nodes}

    if district_type == 'CD':
        fx = '_CD118.txt'
        filepath = filepath_cd
        col = 'CDFP'
    elif district_type == 'SS':
        fx = '_SLDU22.txt'
        filepath = filepath_ss
        col = 'SLDUST'
    elif district_type == 'SH':
        fx = '_SLDL22.txt'
        filepath = filepath_sh
        col = 'SLDLST'
    else:
        raise ValueError(f"Unknown district_type: {district_type}")

    fips = get_fips(state)
    filename = fips + '_' + state + fx
    csv_file = pd.read_csv(filepath + filename, skipinitialspace=True)

    districts = {}
    unassigned = []

    for _, row in csv_file.iterrows():
        g = str(row['GEOID'])
        if len(g) < 15:
            g = '0' + g

        i = geoid_to_node[g]
        j = str(row[col])

        if j in {'ZZ', 'ZZZ'}:
            unassigned.append(i)
        else:
            if j not in districts:
                districts[j] = []
            districts[j].append(i)

    if unassigned:
        print(f"  unassigned = {unassigned}")
    return districts

In [6]:
counts = {"ttt": 0, "ftt": 0, "fft": 0, "fff": 0}

for (state, district_type) in number_of_districts.keys():

    print(f"{'*'*40}")
    print(f"Starting {state} {district_type}")
    print(f"{'*'*40}\n")

    # Read block-level graph
    filename = filepath_graphs + state + "_block.json"
    G = read_graph_from_json(filename, state=state)
    print(f"  nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")

    # Set Euclidean-M edge weights (hierarchical by GEOID20 county/tract)
    set_euclidean_M_weights(G)

    if number_of_districts[state, district_type] <= 1:
        print("  Skipping because k <= 1.")
        continue
    if not nx.is_connected(G):
        print("  Skipping because G is disconnected.")
        continue

    # Read enacted districts from BAF
    districts = read_enacted_districts(G, state, district_type)

    for key in districts:
        district = districts[key]
        population = sum(G.nodes[i]['TOTPOP'] for i in district)
        mean_x = sum(G.nodes[i]['TOTPOP'] * G.nodes[i]['C_X'] for i in district) / population
        mean_y = sum(G.nodes[i]['TOTPOP'] * G.nodes[i]['C_Y'] for i in district) / population

        if not nx.is_connected(G.subgraph(district)):
            print(f"  {key} Skipping — disconnected.")
            continue

        root = nearest_node(G, district, mean_x, mean_y)
        district_set = set(district)
        assert root in district_set

        # --- Weighted shortest paths (Dijkstra) ---
        pred, dist = nx.dijkstra_predecessor_and_distance(G, source=root, weight="weight")

        # Tree-based
        tree_feas = all((i == root or pred[i][0] in district_set) for i in district)

        # Distance-based
        dist_feas = all((i == root or any(j in district_set for j in pred[i])) for i in district)

        # DAG-based
        ordering = sorted(dist.items(), key=lambda item: item[1])
        position = {ordering[p][0]: p for p in range(len(ordering))}
        dag_feas = all(
            i == root or any(
                (j in district_set and position[j] < position[i])
                for j in G.neighbors(i)
            )
            for i in district
        )

        print(f"  {key}  tree={tree_feas}  dist={dist_feas}  dag={dag_feas}")

        # Tally results
        if tree_feas:
            counts["ttt"] += 1
        elif dist_feas:
            counts["ftt"] += 1
        elif dag_feas:
            counts["fft"] += 1
        else:
            counts["fff"] += 1

****************************************
Starting IA CD
****************************************

  nodes: 175199, edges: 416526
  3  tree=False  dist=False  dag=False
  2  tree=False  dist=False  dag=True
  4  tree=False  dist=False  dag=False
  1  tree=False  dist=False  dag=False
****************************************
Starting IA SS
****************************************

  nodes: 175199, edges: 416526
  12  tree=False  dist=False  dag=False
  9  tree=False  dist=False  dag=True
  32  tree=False  dist=False  dag=False
  13  tree=True  dist=True  dag=True
  6  tree=False  dist=False  dag=True
  42  tree=False  dist=False  dag=False
  38  tree=False  dist=False  dag=False
  31  tree=False  dist=False  dag=False
  34  tree=False  dist=False  dag=True
  27  tree=False  dist=False  dag=False
  24  tree=False  dist=False  dag=False
  29  tree=False  dist=False  dag=True
  3  tree=False  dist=False  dag=False
  4  tree=True  dist=True  dag=True
  41  tree=False  dist=False  dag=False
 

In [7]:
# Summarize
total = sum(counts.values())
tree = counts["ttt"]
dist = counts["ftt"] + counts["ttt"]
dag = counts["fft"] + counts["ftt"] + counts["ttt"]

print(f"Counts: {counts}")
print(f"Total districts: {total}")
print()
print(f"Tree-based:     {tree}/{total} -> {round(100*tree/total, 2)}%")
print(f"Distance-based: {dist}/{total} -> {round(100*dist/total, 2)}%")
print(f"DAG-based:      {dag}/{total} -> {round(100*dag/total, 2)}%")

Counts: {'ttt': 8, 'ftt': 0, 'fft': 26, 'fff': 120}
Total districts: 154

Tree-based:     8/154 -> 5.19%
Distance-based: 8/154 -> 5.19%
DAG-based:      34/154 -> 22.08%
